<a href="https://colab.research.google.com/github/gibranfp/CursoAprendizajeProfundo/blob/2027-1/notebooks/3a_celda_recurrente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Celda recurrente básica
En esta libreta programaremos una celda recurrente básica usando tensores de PyTorch.

In [1]:
from collections import Counter
from math import sqrt

import torch as th
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

th.manual_seed(42)

## Ejemplo con oraciones sencillas
Para probar nuestra celda recurrente usaremos dos oraciones pequeñas en español.

In [2]:
textos = ['ya casi termina el mes',
          'ya hay pan de muerto']

n_docs = len(textos)
print(f'Número de documentos: {n_docs}')

Número de documentos: 2


### _Tokenización_
Descomponemos las oraciones en secuencias de caracteres.

In [3]:
tokens = [[c for c in t] for t in textos ]

print(f'Oración 1: {tokens[0]}')
print(f'Oración 2: {tokens[1]}')

Oración 1: ['y', 'a', ' ', 'c', 'a', 's', 'i', ' ', 't', 'e', 'r', 'm', 'i', 'n', 'a', ' ', 'e', 'l', ' ', 'm', 'e', 's']
Oración 2: ['y', 'a', ' ', 'h', 'a', 'y', ' ', 'p', 'a', 'n', ' ', 'd', 'e', ' ', 'm', 'u', 'e', 'r', 't', 'o']


Obtenemos el tamaño máximo de secuencia.

In [4]:
max_sec = max(len(t) for t in tokens)

print(f'Tamaño máximo de secuencia: {max_sec}')

Tamaño máximo de secuencia: 22


### Vocabulario
Construimos dos diccionarios: uno que mapea cada carácter a un índice y el otro que realiza el mapeo inverso.

In [5]:
voc = Counter([p for t in tokens for p in t])

i2p = {i:p for i,(p,f) in enumerate(voc.most_common())}
p2i = {p:i for i,(p,f) in enumerate(voc.most_common())}

tam_voc = len(i2p)

print(f'Tamaño del vocabulario: {tam_voc} caracteres')
print(f'Diccionario de caracteres a índices: {p2i}')
print(f'Diccionario de índices a caracteres: {i2p}')

Tamaño del vocabulario: 17 caracteres
Diccionario de caracteres a índices: {' ': 0, 'a': 1, 'e': 2, 'y': 3, 'm': 4, 's': 5, 'i': 6, 't': 7, 'r': 8, 'n': 9, 'c': 10, 'l': 11, 'h': 12, 'p': 13, 'd': 14, 'u': 15, 'o': 16}
Diccionario de índices a caracteres: {0: ' ', 1: 'a', 2: 'e', 3: 'y', 4: 'm', 5: 's', 6: 'i', 7: 't', 8: 'r', 9: 'n', 10: 'c', 11: 'l', 12: 'h', 13: 'p', 14: 'd', 15: 'u', 16: 'o'}


### Conversión a secuencias de vectores 1-de-K
Primero convertimos las oraciones de secuencias de _tokens_ a secuencias de índices.

In [6]:
indsecs = [[p2i[c] for c in t] for t in tokens]

print(f'Oración 1 (tam={len(indsecs[0])}): {indsecs[0]}')
print(f'Oración 2 (tam={len(indsecs[1])}): {indsecs[1]}')

Oración 1 (tam=22): [3, 1, 0, 10, 1, 5, 6, 0, 7, 2, 8, 4, 6, 9, 1, 0, 2, 11, 0, 4, 2, 5]
Oración 2 (tam=20): [3, 1, 0, 12, 1, 3, 0, 13, 1, 9, 0, 14, 2, 0, 4, 15, 2, 8, 7, 16]


Guardamos las longitudes originales, ya que al formar el tensor rellenaremos la oración más corta con vectores de ceros.

In [7]:
longitudes = th.tensor([len(s) for s in indsecs])

print(f'Longitudes: {longitudes.tolist()}')

Longitudes: [22, 20]


Posteriormente generamos un tensor con las secuencias de vectores 1-de-K.

In [8]:
ohesecs = th.zeros((n_docs, max_sec, tam_voc))

for i,s in enumerate(indsecs):
  for j,c in enumerate(s):
    ohesecs[i, j, c] = 1

print(f'Tensor con oraciones: {ohesecs}')

Tensor con oraciones: tensor([[[0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
         [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 

### Procesamiento de secuencias con una celda recurrente
Una celda recurrente básica consiste en una capa densa que se retroalimenta. Al procesar una secuencia, la celda recurrente básica tiene como entrada en el tiempo $t$:
* Entrada actual $\mathbf{x}^{[t]}$
* Estado anterior $\mathbf{h}^{[t-1]}$

Para el tiempo $t$, la salida de esta capa es el estado $\mathbf{h}^{[t]}$, el cual se obtiene como:

$$
\begin{align*}
  \mathbf{h}^{[t]} & = \tanh\left(\mathbf{W}_{hh}\cdot \mathbf{h}^{[t-1]} + \mathbf{W}_{xh}\cdot \mathbf{x}^{[t]} + \mathbf{b}_h\right)\\
\end{align*}
$$

donde $\mathbf{W}_{hh} \in \mathbb{R}^{r\times r}$, $\mathbf{W}_{xh} \in \mathbb{R}^{r\times d}$, $\mathbf{b}_h \in \mathbb{R}^{r\times 1}$, $r$ es la dimensión del estado y $d$ la de la entrada (en nuestro caso el tamaño del vocabulario).

De forma equivalente, podemos ver esta operación como:

$$
\begin{align*}
  \mathbf{h}^{[t]} & = \tanh\left(\mathbf{W}_h \cdot \underbrace{\left[\mathbf{h}^{[t-1]}; \mathbf{x}^{[t]}\right]}_{\text{Concatenación}} + \mathbf{b}_h\right)\\
\end{align*}
$$

donde $\mathbf{W}_h = [\mathbf{W}_{hh}, \mathbf{W}_{xh}] \in \mathbb{R}^{r\times (r + d)}$.

Creamos un estado inicial e inicializamos las matrices de pesos y el vector de sesgos de la celda. Representamos la entrada como un tensor unidimensional de forma $(d,)$ y el estado y los sesgos como tensores de forma $(r,)$, en lugar de vectores columna. El producto `@` entre una matriz y un tensor unidimensional funciona igual que con un vector columna y podemos usar directamente cada `ohesecs[i, t]` sin cambiar su forma.

In [9]:
tam_edo = 10

h = th.zeros((tam_edo))

k = 1 / tam_edo
Wxh = th.Tensor(tam_edo, tam_voc).uniform_(-sqrt(k), sqrt(k))
Whh = th.Tensor(tam_edo, tam_edo).uniform_(-sqrt(k), sqrt(k))
bh = th.Tensor(tam_edo).uniform_(-sqrt(k), sqrt(k))

print(f'W_xh={Wxh.shape}')
print(f'W_hh={Whh.shape}')
print(f'b={bh.shape}')
print(f'h[0]={h}')

W_xh=torch.Size([10, 17])
W_hh=torch.Size([10, 10])
b=torch.Size([10])
h[0]=tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


Procesamos una de las secuencias, obteniendo cada nuevo estado a partir del estado anterior y la entrada actual.

Paso 1:

In [10]:
print(h)

x = ohesecs[0, 0]
h = th.tanh(Whh @ h + Wxh @ x + bh)

print(x)
print(h)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
tensor([ 0.4155, -0.4298,  0.4234,  0.3164,  0.3837, -0.2300, -0.4429,  0.0491,
         0.1052, -0.1447])


Paso 2:

In [11]:
x = ohesecs[0, 1]
h = th.tanh(Whh @ h + Wxh @ x + bh)

print(h)

tensor([ 0.4915, -0.5052,  0.5205,  0.1585, -0.1289, -0.3857, -0.4621,  0.1931,
        -0.2121,  0.2424])


Paso 3:

In [12]:
x = ohesecs[0, 2]
h = th.tanh(Whh @ h + Wxh @ x + bh)

print(h)

tensor([ 0.4410, -0.4023,  0.2849,  0.3349, -0.2405, -0.5442, -0.4909,  0.2337,
        -0.0683, -0.3638])


## Clase de celda recurrente (estado y entrada separados)
Definimos una clase con la celda recurrente básica con la entrada y el estado tomados como vectores separados y procesados con matrices de pesos separadas.

In [13]:
class CeldaRecurrente:
  def __init__(self,
               d_ent: int,
               d_edo: int,
               disp: str = 'cpu') -> None:
    self.input_size = d_ent
    self.hidden_size = d_edo
    self.disp = disp

    k = sqrt(1 / d_edo)
    self.Wxh = th.empty(d_edo, d_ent, device=disp).uniform_(-k, k)
    self.Whh = th.empty(d_edo, d_edo, device=disp).uniform_(-k, k)
    self.bh = th.zeros((d_edo), device=disp)

  def __call__(self,
               x: th.Tensor,
               h: th.Tensor) -> th.Tensor:
    h = th.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
    return h

  def to(self,
         disp: str) -> None:
    self.Wxh = self.Wxh.to(disp)
    self.Whh = self.Whh.to(disp)
    self.bh = self.bh.to(disp)
    self.disp = disp

Definimos una función para procesar múltiples secuencias paso a paso con una celda recurrente. Para cada secuencia, partimos del estado inicial $\mathbf{h}^{[0]}$ y aplicamos la celda solo hasta su longitud real, de modo que las posiciones de relleno no se procesan. En nuestro caso, la segunda oración tiene 20 caracteres y está rellenada con dos vectores de ceros, por lo que si no hiciéramos esta distinción, la celda los procesaría como si fueran caracteres y el estado final no correspondería al último carácter real.

In [14]:
def ejecuta_celda(celda, X, h0, longitudes):
  # Calculamos los estados para todas las entradas
  for i in range(X.shape[0]):
    print(f'Texto {i}')
    h = h0 # primer estado con ceros
    print(f'\tEstado 0: {h}')
    for t in range(longitudes[i]):
      print(f'\tEntrada {t}: {X[i, t]}')
      h = celda(X[i, t], h)
      print(f'\tEstado {t + 1}: {h}')

Instanciamos nuestra celda y procesamos las secuencias.

In [15]:
disp = 'cuda:0' if th.cuda.is_available() else 'cpu'

ohesecs = ohesecs.to(disp)
h0 = th.zeros((tam_edo), device=disp)

celdarec = CeldaRecurrente(tam_voc, tam_edo, disp=disp)
ejecuta_celda(celdarec, ohesecs, h0, longitudes)

Texto 0
	Estado 0: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')
	Entrada 0: tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 1: tensor([-0.2828, -0.3019, -0.1818, -0.1543,  0.2657,  0.1566, -0.1472,  0.2000,
         0.2798,  0.2787], device='cuda:0')
	Entrada 1: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 2: tensor([-0.1974,  0.0957,  0.0256,  0.1361,  0.0083,  0.0132, -0.0716, -0.0952,
         0.2553, -0.0368], device='cuda:0')
	Entrada 2: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 3: tensor([-0.0117, -0.0799, -0.1665,  0.2662,  0.2201, -0.0934, -0.0875,  0.0620,
        -0.3314, -0.3109], device='cuda:0')
	Entrada 3: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 4: tensor([-0.2195, -0.1361,  0.4709, -0.2107, -0.

## Clase de celda recurrente (estado y entrada juntos)
Definimos una clase con la celda recurrente básica con el estado y la entrada tomados como un solo vector y procesados por una sola matriz de pesos.

In [16]:
class CeldaRecurrenteConcat:
  def __init__(self,
               d_ent: int,
               d_edo: int,
               disp: str = "cpu") -> None:
    self.input_size = d_ent
    self.hidden_size = d_edo
    self.disp = disp

    k = sqrt(1 / (d_edo + d_ent))
    self.Wh = th.empty(d_edo, d_edo + d_ent, device=disp).uniform_(-k, k)
    self.bh = th.zeros((d_edo), device=disp)

  def __call__(self,
               x: th.Tensor,
               h: th.Tensor) -> th.Tensor:
    hx = th.concat([h, x], dim=0)
    h = th.tanh(self.Wh @ hx + self.bh)
    return h

  def to(self,
         disp: str) -> None:
    self.Wh = self.Wh.to(disp)
    self.bh = self.bh.to(disp)
    self.disp = disp

celda_recconcat = CeldaRecurrenteConcat(tam_voc, tam_edo, disp=disp)
ejecuta_celda(celda_recconcat, ohesecs, h0, longitudes)

Texto 0
	Estado 0: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')
	Entrada 0: tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 1: tensor([-0.0848,  0.1735, -0.1533, -0.1593,  0.0526, -0.1542,  0.0228, -0.1451,
        -0.0548, -0.0533], device='cuda:0')
	Entrada 1: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 2: tensor([-0.0899,  0.1071, -0.0373, -0.0584,  0.1137,  0.2276,  0.0041, -0.0208,
         0.1624,  0.0899], device='cuda:0')
	Entrada 2: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 3: tensor([-0.0047,  0.1486, -0.0343, -0.1335,  0.0548, -0.0688, -0.0103, -0.1088,
         0.1408,  0.0316], device='cuda:0')
	Entrada 3: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 4: tensor([-0.0221,  0.1763, -0.0968, -0.0460,  0.

## Clase `RNNCell` de PyTorch
PyTorch cuenta con la clase `RNNCell`, la cual implementa una celda recurrente básica. Esta clase usa dos vectores de sesgo y sus matrices de pesos se inicializan de forma distinta.

In [17]:
recth = th.nn.RNNCell(tam_voc, tam_edo)
recth.to(disp)

ejecuta_celda(recth, ohesecs, h0, longitudes)

Texto 0
	Estado 0: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='cuda:0')
	Entrada 0: tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 1: tensor([-0.0105,  0.4320,  0.2378,  0.0334, -0.1171, -0.6083,  0.1369, -0.0846,
        -0.3424, -0.0999], device='cuda:0', grad_fn=<SqueezeBackward1>)
	Entrada 1: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 2: tensor([-0.3116,  0.3188,  0.0940, -0.3949, -0.1175, -0.4606,  0.1714,  0.0109,
        -0.1112, -0.3605], device='cuda:0', grad_fn=<SqueezeBackward1>)
	Entrada 2: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       device='cuda:0')
	Estado 3: tensor([-0.0569,  0.4122,  0.1445, -0.1915, -0.3899, -0.5101,  0.1373, -0.1642,
        -0.5485,  0.2152], device='cuda:0', grad_fn=<SqueezeBackward1>)
	Entrada 3: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]

PyTorch también cuenta con la clase `RNN`, una interfaz de más alto nivel que `RNNCell` que puede procesar varios pasos y ejemplos a la vez. Además, permite apilar celdas (capas recurrentes) y crear redes bidireccionales que procesan la secuencia en ambos sentidos.

Para procesar secuencias de distinta longitud, las empaquetamos con `pack_padded_sequence`, indicando la longitud real de cada una. Así, el estado final de cada secuencia corresponde a su último elemento válido y no al relleno. Después, `pad_packed_sequence` reconstruye el tensor con relleno a partir de las salidas empaquetadas.

In [18]:
rnn = th.nn.RNN(tam_voc, tam_edo, batch_first=True)
rnn.to(disp)

empaquetado = pack_padded_sequence(ohesecs, longitudes,
                                   batch_first=True, enforce_sorted=False)
outputs_emp, h = rnn(empaquetado)
outputs, _ = pad_packed_sequence(outputs_emp, batch_first=True)

print(f'outputs.shape={outputs.shape}')
print(f'h.shape={h.shape}')

outputs.shape=torch.Size([2, 22, 10])
h.shape=torch.Size([1, 2, 10])


Comprobamos que el último estado de la segunda oración (de longitud 20) es el del último carácter real.

In [19]:
print(th.allclose(h[0, 1], outputs[1, longitudes[1] - 1]))

True
